In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

torch.cuda.is_available()

True

In [25]:
512 // 2, 512 % 2

(256, 0)

In [258]:
from next_sequence.core import TaskType, Feature, CoreDataset
from next_sequence.collate_fn import CustomCollate
from next_sequence.utils import create_bow
from next_sequence.embedding import EmbeddingModel
from next_sequence.fusion import BaseFusion
from next_sequence.create_cls import PrependCLS
from next_sequence.attention_head import MultiHeadAttention
from next_sequence.position_embedding import RoPE, RoPEStrategy
from next_sequence.encoder_block import EncoderBlock, EncoderCLSBackbone
from typing import Any
from next_sequence.multi_task_head import MultiTaskHeads
from next_sequence.multi_task_model import MultiTaskModel
import torch.nn as nn

# Constant

In [259]:
D_MODEL = 512
N_HEADS = 2

# Mock Dataset

In [260]:
import pandas as pd

df = pd.DataFrame({
    'id': [1,2,3],
    'item': [['a','a','b'], ['a','b'], ['a','a','a','a','b']],
    'age': [[25,26,27], [35,37], [41,41,42,42,55]],
    'price': [[100, 110, 120], [1000, 1500], [1000, 1000, 1000, 1000, 2500]],
})

df.head()

,id,item,age,price
0,1,"[a, a, b]","[25, 26, 27]","[100, 110, 120]"
1,2,"[a, b]","[35, 37]","[1000, 1500]"
2,3,"[a, a, a, a, b]","[41, 41, 42, 42, 55]","[1000, 1000, 1000, 1000, 2500]"


In [261]:
item_bow, item_vocab_size, max_len = create_bow(df['item'].tolist())
df['item'] = df['item'].apply(lambda x: [item_bow[_x] for _x in x])
df

,id,item,age,price
0,1,"[2, 2, 1]","[25, 26, 27]","[100, 110, 120]"
1,2,"[2, 1]","[35, 37]","[1000, 1500]"
2,3,"[2, 2, 2, 2, 1]","[41, 41, 42, 42, 55]","[1000, 1000, 1000, 1000, 2500]"


# Create batch

In [262]:
samples = df.to_dict(orient='records')
dataset = CoreDataset(samples)
batch = dataset[:]
batch

[{'id': 1, 'item': [2, 2, 1], 'age': [25, 26, 27], 'price': [100, 110, 120]},
 {'id': 2, 'item': [2, 1], 'age': [35, 37], 'price': [1000, 1500]},
 {'id': 3,
  'item': [2, 2, 2, 2, 1],
  'age': [41, 41, 42, 42, 55],
  'price': [1000, 1000, 1000, 1000, 2500]}]

# Custom Collate (you can apply to your case)

In [263]:
# all custom collate_fn here will be changed project by project
def _pack_xy(b, dtype):
    return torch.tensor(b, dtype=dtype)

def collate_common(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None
    
    x = [_pack_xy(b[:-1], torch.long) for b in batch]
    y = [_pack_xy(b[-1], torch.long) for b in batch]
    return x, y

def collate_delta(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.float) for b in batch], None

    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(b[-1]-b[-2], torch.float) for b in batch]
    return x, y

def collate_delta_bin(batch, inference=False):
    if inference:
        return [_pack_xy(b, torch.long) for b in batch], None

    def logic(b):
        if b <= 1:
            return 1
        if b <= 3:
            return 2
        return 3
    # if embedding is linear, x dtype must be float only
    x = [_pack_xy(b[:-1], torch.float) for b in batch]
    y = [_pack_xy(logic(b[-1]-b[-2]), torch.long) for b in batch]
    return x, y

In [264]:
features = [
    Feature(name='item', task=TaskType.CLASSIFICATION, vocab_size=item_vocab_size),
    Feature(name='age', task=TaskType.CLASSIFICATION, vocab_size=3+1, is_linear=True),
    Feature(name='price', task=TaskType.REGRESSION, is_linear=True),
]
funcs = {
    'item': collate_common,
    'age': collate_delta_bin,
    'price': collate_delta
}
collate_fn = CustomCollate(features, 'item', funcs, inference=False)

In [265]:
prep_data = collate_fn(batch)
prep_data

{'X': defaultdict(list,
             {'item': tensor([[2, 2, 0, 0],
                      [2, 0, 0, 0],
                      [2, 2, 2, 2]]),
              'age': tensor([[[25.],
                       [26.],
                       [ 0.],
                       [ 0.]],
              
                      [[35.],
                       [ 0.],
                       [ 0.],
                       [ 0.]],
              
                      [[41.],
                       [41.],
                       [42.],
                       [42.]]]),
              'price': tensor([[[ 100.],
                       [ 110.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [   0.],
                       [   0.],
                       [   0.]],
              
                      [[1000.],
                       [1000.],
                       [1000.],
                       [1000.]]])}),
 'Y': defaultdict(list,
  

# Embedding

In [266]:
embedder = EmbeddingModel(features=features, d_model=D_MODEL)

In [267]:
prep_data['X']['item']

tensor([[2, 2, 0, 0],
        [2, 0, 0, 0],
        [2, 2, 2, 2]])

In [268]:
vectors = embedder(prep_data['X'])
for feature in features:
    print(vectors[feature.name].shape)

torch.Size([3, 4, 512])
torch.Size([3, 4, 512])
torch.Size([3, 4, 512])


# Fusion Strategy (you can apply to your case)

In [269]:
class SumFusion(BaseFusion):
    def forward(self, X:dict[str, Any])->Any:
        return torch.stack([
            X[feature.name]
            for feature in self.features
        ]).sum(0)

    def output_dim(self)->int:
        return self.d_model

class ConcatFusion(BaseFusion):
    def forward(self, X:dict[str, Any])->Any:
        return torch.cat([X[feature.name] for feature in self.features], dim=-1)
    def output_dim(self)->int:
        return len(self.features) * self.d_model

class ConcatProjectionFusion(BaseFusion):
    def __init__(self, features, d_model, d_out=None):
        self._target = d_out or d_model
        super().__init__(features, d_model)
        self.proj = nn.Linear(len(features) * d_model, self._target)
    def forward(self, X):
        return self.proj(torch.cat([X[f.name] for f in self.features], dim=-1))
    def output_dim(self):
        return self._target

In [270]:
fusion = SumFusion(features=features, d_model=D_MODEL)
fusion.d_out, fusion(vectors).shape

(512, torch.Size([3, 4, 512]))

In [271]:
fusion = ConcatFusion(features=features, d_model=D_MODEL)
fusion.d_out, fusion(vectors).shape

(1536, torch.Size([3, 4, 1536]))

In [272]:
fusion = ConcatProjectionFusion(features=features, d_model=D_MODEL, d_out=int(D_MODEL/2))
fusion.d_out, fusion(vectors).shape

(256, torch.Size([3, 4, 256]))

In [273]:
prep_data['mask']

tensor([[False, False,  True,  True],
        [False,  True,  True,  True],
        [False, False, False, False]])

In [274]:
fusion = SumFusion(features=features, d_model=D_MODEL)
fused_vectors = fusion(vectors)
fused_vectors.shape

torch.Size([3, 4, 512])

# Prepend CLS which will be used later in the multi-task prediction

In [275]:
prepend_cls = PrependCLS(d_model=fusion.d_model)
x, mask = prepend_cls(fused_vectors, prep_data['mask'])

In [276]:
x.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5]))

In [277]:
mask

tensor([[False, False, False,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False, False, False]])

# Attention Head with RoPE

In [278]:
rope = RoPE(fusion.d_out // N_HEADS, max_len, RoPEStrategy.INTERLEAVED)

In [279]:
attn = MultiHeadAttention(d_model=fusion.d_model, n_heads=N_HEADS, rope=rope)

In [280]:
out = attn(x, mask)
out.shape

torch.Size([3, 5, 512])

In [281]:
x.shape, out.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5, 512]), torch.Size([3, 5]))

# Test EncoderCLSBlock (combine, rope, attn together)

In [282]:
block = EncoderBlock(d_model=fusion.d_out, n_heads=N_HEADS, rope=rope, expansion=4, dropout=0.1)

In [283]:
b_x = block(x, mask)
x.shape, b_x.shape, mask.shape

(torch.Size([3, 5, 512]), torch.Size([3, 5, 512]), torch.Size([3, 5]))

In [284]:
emb = EmbeddingModel(features, D_MODEL)
fusion = SumFusion(features, D_MODEL)
prepend_cls = PrependCLS(fusion.d_out)
rope = RoPE(fusion.d_out // N_HEADS, max_len + 1)
backbone = EncoderCLSBackbone(emb, fusion, prepend_cls, rope, N_HEADS)

In [285]:
backbone.eval()
with torch.no_grad():
    representation = backbone(prep_data['X'], prep_data['mask'])
representation.shape

torch.Size([3, 512])

In [286]:
representation

tensor([[ 0.9655,  1.3421, -0.7628,  ...,  0.0708, -0.8278,  0.1995],
        [ 0.7826,  1.1092, -0.0198,  ...,  0.2375, -0.6547,  0.0027],
        [ 0.9667,  1.1318, -1.3129,  ..., -0.4882, -1.0790,  0.4463]])

# Predict with MultiTaskHeads

In [287]:
mlh = MultiTaskHeads(features=features, d_model=fusion.d_out)

In [288]:
mlh.eval()
with torch.no_grad():
    predictions = mlh(representation)

In [289]:
predictions.keys()

dict_keys(['item', 'age', 'price'])

In [290]:
predictions['item'].softmax(dim=-1)

tensor([[0.1950, 0.3071, 0.4980],
        [0.2378, 0.2979, 0.4643],
        [0.1903, 0.4035, 0.4062]])

In [291]:
predictions['age'].softmax(dim=-1)

tensor([[0.0477, 0.1210, 0.6000, 0.2313],
        [0.0392, 0.1686, 0.5895, 0.2026],
        [0.0556, 0.0968, 0.6097, 0.2379]])

In [292]:
predictions['price']

tensor([[-0.6257],
        [-0.2995],
        [-0.7757]])

# Model combined backbone, mlh

In [293]:
model = MultiTaskModel(backbone, mlh)

In [294]:
model.eval()
with torch.no_grad():
    _predictions = model(prep_data['X'], prep_data['mask'])

In [295]:
_predictions['item'].softmax(dim=-1)

tensor([[0.1950, 0.3071, 0.4980],
        [0.2378, 0.2979, 0.4643],
        [0.1903, 0.4035, 0.4062]])

In [296]:
_predictions['age'].softmax(dim=-1)

tensor([[0.0477, 0.1210, 0.6000, 0.2313],
        [0.0392, 0.1686, 0.5895, 0.2026],
        [0.0556, 0.0968, 0.6097, 0.2379]])

In [297]:
_predictions['price']

tensor([[-0.6257],
        [-0.2995],
        [-0.7757]])

In [298]:
model.eval()
with torch.no_grad():
    rep = backbone(prep_data['X'], prep_data['mask'])
    manual = mlh(rep)
    wrapped = model(prep_data['X'], prep_data['mask'])

all(torch.allclose(manual[k], wrapped[k], atol=1e-6) for k in manual)   # expect True

True

In [299]:
from next_sequence.loss_fn import UncertaintyWeightedLoss

In [300]:
features

[Feature(name='item', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=3, is_linear=False),
 Feature(name='age', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=4, is_linear=True),
 Feature(name='price', task=<TaskType.REGRESSION: 'regression'>, vocab_size=0, is_linear=True)]

In [301]:
wrapped

{'item': tensor([[-0.6858, -0.2317,  0.2518],
         [-0.4802, -0.2548,  0.1891],
         [-0.7571, -0.0052,  0.0015]]),
 'age': tensor([[-1.6844, -0.7547,  0.8467, -0.1066],
         [-1.7750, -0.3171,  0.9345, -0.1334],
         [-1.5551, -0.9998,  0.8403, -0.1008]]),
 'price': tensor([[-0.6257],
         [-0.2995],
         [-0.7757]])}

In [302]:
prep_data['Y']

defaultdict(list,
            {'item': tensor([1, 1, 1]),
             'age': tensor([1, 2, 3]),
             'price': tensor([  10.,  500., 1500.])})

In [303]:
features

[Feature(name='item', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=3, is_linear=False),
 Feature(name='age', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=4, is_linear=True),
 Feature(name='price', task=<TaskType.REGRESSION: 'regression'>, vocab_size=0, is_linear=True)]

In [304]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

In [305]:
prep_data['Y']

defaultdict(list,
            {'item': tensor([1, 1, 1]),
             'age': tensor([1, 2, 3]),
             'price': tensor([  10.,  500., 1500.])})

In [306]:
feature = features[0]
loss_fn(wrapped[feature.name], prep_data['Y'][feature.name])

tensor(1.0998)

In [307]:
ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()

losses = {}
for feature in features:
    pred = wrapped[feature.name]
    target = prep_data['Y'][feature.name]
    if feature.task == TaskType.CLASSIFICATION:
        losses[feature.name] = ce(pred, target)
    else:
        losses[feature.name] = mse(pred.squeeze(-1), target.float())

# loss = sum(losses.values())
sum(losses.values())


tensor(834249.1875)

In [308]:
for feature in features:
    print(losses[feature.name])

tensor(1.0998)
tensor(1.3589)
tensor(834246.7500)


In [ ]:
# import torch
# import torch.nn as nn
# from next_sequence.core import Feature, TaskType


# class UncertaintyWeightedLoss(nn.Module):
#     """Kendall, Gal & Cipolla (2018): multi-task loss weighting by
#     homoscedastic uncertainty. Learns s = log(sigma^2) per task.

#       regression:      0.5 * exp(-s) * L_mse + 0.5 * s
#       classification:        exp(-s) * L_ce  + 0.5 * s
#     """

#     def __init__(self, features: list[Feature]):
#         super().__init__()
#         self.features = features
#         self.log_vars = nn.ParameterDict(
#             {f.name: nn.Parameter(torch.zeros(())) for f in features}
#         )

#     def forward(self, losses: dict[str, torch.Tensor]) -> torch.Tensor:
#         terms = []
#         for f in self.features:
#             s = self.log_vars[f.name]
#             precision = torch.exp(-s)
#             if f.task == TaskType.CLASSIFICATION:
#                 terms.append(precision * losses[f.name] + 0.5 * s)
#             else:  # regression
#                 terms.append(0.5 * precision * losses[f.name] + 0.5 * s)
#         return torch.stack(terms).sum()

#     @torch.no_grad()
#     def weights(self) -> dict[str, float]:
#         """Effective multiplier on each task's loss."""
#         return {
#             f.name: (1.0 if f.task == TaskType.CLASSIFICATION else 0.5)
#                     * torch.exp(-self.log_vars[f.name]).item()
#             for f in self.features
#         }

In [329]:
from next_sequence.loss_fn import UncertaintyWeightedLoss

ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
criterion = UncertaintyWeightedLoss(features).to(device)
# optimizer = torch.optim.Adam(list(model.parameters()) + list(criterion.parameters()), lr=1e-3)
optimizer = torch.optim.Adam([
    {"params": model.parameters(),     "lr": 1e-3},
    {"params": criterion.parameters(), "lr": 1e-2},  # log_vars adapt faster
])

# per step
losses = {}
for feature in features:
    pred = wrapped[feature.name]
    target = prep_data['Y'][feature.name]
    if feature.task == TaskType.CLASSIFICATION:
        losses[feature.name] = ce(pred, target)
    else:
        losses[feature.name] = mse(pred.squeeze(-1), target.float())

loss = criterion(losses)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# monitor: raw losses + effective weights, not just the total
print({k: v.item() for k, v in losses.items()}, criterion.weights())

{'item': 1.099768877029419, 'age': 1.358858585357666, 'price': 834246.75} {'item': 0.9900498390197754, 'age': 0.9900498390197754, 'price': 0.4950249195098877}


In [330]:
losses.values()

dict_values([tensor(1.0998), tensor(1.3589), tensor(834246.7500)])

In [314]:
nn.Parameter(torch.zeros(3))

Parameter containing:
tensor([0., 0., 0.], requires_grad=True)

In [315]:
nn.ParameterDict({feature.name:nn.Parameter(torch.zeros(0)) for feature in features})['item']

Parameter containing:
tensor([], requires_grad=True)

In [316]:
criterion.log_vars['item']

Parameter containing:
tensor(0., requires_grad=True)

In [317]:
torch.exp(-criterion.log_vars['item'])

tensor(1., grad_fn=<ExpBackward0>)

In [318]:
torch.exp(-torch.tensor(0)), torch.exp(torch.tensor(0))

(tensor(1.), tensor(1.))

In [319]:
# criterion.precisions()
criterion.weights()

{'item': 1.0, 'age': 1.0, 'price': 0.5}